# 05 — Evaluasi Pipeline (Notebook Terpenting)
**SuaraLens** | Evaluasi menyeluruh pipeline klasifikasi, urgency scoring, dan sentimen.

Notebook ini menghasilkan metrik formal untuk laporan akademik.
Pastikan NB 03 (llm_engine.py) dan NB 04 (sentiment_model.py) sudah selesai.


In [ ]:
import sys
sys.path.insert(0, '..')

import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from modules.llm_engine   import classify_llm, check_ollama_status, VALID_CATEGORIES
from modules.sentiment_model import classify_sentiment_batch, load_sentiment_model

sns.set_theme(style='whitegrid')
DATA_PATH   = '../data/suaralens_dummy_simulasi.jsonl'
OUTPUT_PATH = '../data/output/evaluation_report.json'

df = pd.read_json(DATA_PATH, lines=True)
print(f'Dataset: {len(df):,} baris')
print(f'Ollama status: {check_ollama_status()["running"]}')


## 1. Stratified Sample 500 Baris

In [ ]:
N_EVAL = 500

sample_df = (
    df.groupby('kategori_true', group_keys=False)
      .apply(lambda x: x.sample(
          max(1, int(N_EVAL * len(x) / len(df))), random_state=42
      ))
      .head(N_EVAL)
      .reset_index(drop=True)
)

print(f'Sampel evaluasi: {len(sample_df)} baris')
print('Distribusi kategori:')
print(sample_df['kategori_true'].value_counts().to_string())


## 2. Evaluasi Kategori — LLM Classification

In [ ]:
print('Menjalankan classify_llm pada 500 sampel...')
print('(Estimasi: 10-30 menit tergantung hardware — pertimbangkan jalankan overnight)')

status = check_ollama_status()
model  = status.get('recommended_model')

llm_results = []
for i, row in sample_df.iterrows():
    result = classify_llm(row['teks_aduan'], model=model)
    llm_results.append({
        'id_aduan':         row['id_aduan'],
        'kategori_true':    row['kategori_true'],
        'urgency_true':     row['urgency_label_true'],
        'urgency_score_true': row['urgency_score_true'],
        'kategori_pred':    result.get('kategori')       if result else None,
        'urgency_pred':     result.get('urgency_label')  if result else None,
        'urgency_score_pred': result.get('urgency_score') if result else None,
        'confidence':       result.get('confidence')     if result else None,
        'error':            result.get('error')          if result else 'None',
    })
    if (i + 1) % 50 == 0:
        print(f'  Progress: {i+1}/{len(sample_df)}')

df_llm = pd.DataFrame(llm_results)
df_valid = df_llm[df_llm['kategori_pred'].notna()]
print(f'\nBerhasil: {len(df_valid)}/{len(df_llm)} ({len(df_valid)/len(df_llm)*100:.1f}%)')


In [ ]:
print('=== Classification Report — Kategori ===')
report = classification_report(
    df_valid['kategori_true'],
    df_valid['kategori_pred'],
    labels=VALID_CATEGORIES,
    output_dict=True,
    zero_division=0
)
print(classification_report(
    df_valid['kategori_true'],
    df_valid['kategori_pred'],
    labels=VALID_CATEGORIES,
    zero_division=0
))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(
    df_valid['kategori_true'],
    df_valid['kategori_pred'],
    labels=VALID_CATEGORIES
)
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=VALID_CATEGORIES, yticklabels=VALID_CATEGORIES,
            linewidths=0.5, ax=ax)
ax.set_title('Confusion Matrix — Klasifikasi Kategori')
ax.set_xlabel('Prediksi')
ax.set_ylabel('Ground Truth')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 3. Uji Self-Consistency Urgency (30 teks × 5 run)

In [ ]:
N_CONSISTENCY = 30
N_RUNS        = 5

consistency_sample = sample_df.sample(N_CONSISTENCY, random_state=99).reset_index(drop=True)
consistency_results = []

for i, row in consistency_sample.iterrows():
    runs = []
    for run in range(N_RUNS):
        res = classify_llm(row['teks_aduan'], model=model)
        runs.append({
            'urgency_label': res.get('urgency_label') if res else None,
            'urgency_score': res.get('urgency_score') if res else None,
        })
    scores = [r['urgency_score'] for r in runs if r['urgency_score'] is not None]
    labels = [r['urgency_label'] for r in runs if r['urgency_label'] is not None]

    consistency_results.append({
        'id_aduan':          row['id_aduan'],
        'urgency_true':      row['urgency_label_true'],
        'runs':              runs,
        'score_std':         round(float(np.std(scores)), 4) if scores else None,
        'score_mean':        round(float(np.mean(scores)), 4) if scores else None,
        'exact_agreement':   len(set(labels)) == 1 if labels else False,
    })

df_consistency = pd.DataFrame(consistency_results)
exact_rate = df_consistency['exact_agreement'].mean() * 100
avg_std    = df_consistency['score_std'].mean()

print(f'=== Self-Consistency Urgency Scoring ===')
print(f'  Sampel teks    : {N_CONSISTENCY}')
print(f'  Runs per teks  : {N_RUNS}')
print(f'  Exact agreement: {exact_rate:.1f}% (label sama di semua {N_RUNS} run)')
print(f'  Avg std score  : {avg_std:.4f}')
print()
print('Interpretasi:')
print('  Exact agreement tinggi (>80%) = model konsisten dan dapat dipercaya')
print('  Avg std rendah (<0.05) = variasi skor antar-run minimal')


## 4. Evaluasi Sentimen

In [ ]:
print('Menjalankan classify_sentiment pada 500 sampel...')
load_sentiment_model()
texts      = df_valid['id_aduan'].map(df.set_index('id_aduan')['teks_aduan']).tolist()
sent_true  = df_valid['id_aduan'].map(df.set_index('id_aduan')['sentiment_true']).tolist()
sent_preds = classify_sentiment_batch(texts)
sent_pred  = [p['sentiment'] for p in sent_preds]

print('\n=== Classification Report — Sentimen ===')
print(classification_report(sent_true, sent_pred,
      target_names=['negative', 'neutral', 'positive'], zero_division=0))
print()
print('⚠ CATATAN PENTING: sentiment_true adalah label sintetis dari generator data,')
print('  BUKAN anotasi manusia. Akurasi di atas harus dibaca hati-hati.')


## 5. Confidence-Based Routing Analysis

In [ ]:
# Threshold routing
CONF_THRESHOLD = 0.75
HIGH_URGENCY   = ['High', 'Critical']

manual_review = df_llm[
    (df_llm['confidence'].fillna(0) < CONF_THRESHOLD) |
    (df_llm['urgency_pred'].isin(HIGH_URGENCY))
]

routing_pct = len(manual_review) / len(df_llm) * 100

print(f'=== Confidence-Based Routing ===')
print(f'  Threshold confidence : {CONF_THRESHOLD}')
print(f'  Total sampel evaluasi: {len(df_llm)}')
print(f'  Masuk antrian manual : {len(manual_review)} ({routing_pct:.1f}%)')
print()
print('Analisis:')
if routing_pct < 15:
    print('  ✓ Routing workload realistis — tidak terlalu membebani reviewer')
elif routing_pct > 50:
    print('  ⚠ Routing terlalu banyak — pertimbangkan naikkan threshold atau fine-tune model')
else:
    print('  ✓ Routing dalam batas wajar')


## 6. Ringkasan Metrik — Siap untuk Laporan

In [ ]:
category_accuracy = accuracy_score(
    df_valid['kategori_true'],
    df_valid['kategori_pred']
) if len(df_valid) > 0 else 0

sentiment_accuracy = accuracy_score(sent_true, sent_pred)

summary_metrics = {
    'generated_at':           pd.Timestamp.now().isoformat(),
    'eval_sample_size':       len(sample_df),
    'valid_llm_responses':    len(df_valid),
    'category': {
        'accuracy':           round(category_accuracy * 100, 2),
        'classification_report': {k: v for k, v in report.items() if k in VALID_CATEGORIES or k in ['accuracy', 'macro avg', 'weighted avg']},
    },
    'urgency_consistency': {
        'n_texts':            N_CONSISTENCY,
        'n_runs':             N_RUNS,
        'exact_agreement_pct': round(exact_rate, 2),
        'avg_score_std':      round(float(avg_std), 4),
    },
    'sentiment': {
        'accuracy':           round(sentiment_accuracy * 100, 2),
        'caveat':             'Label sintetis — bukan ground truth manusia',
    },
    'routing': {
        'threshold':          CONF_THRESHOLD,
        'manual_review_pct':  round(routing_pct, 2),
    },
    'detail_results':         llm_results[:50],  # 50 sampel untuk referensi
}

output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(summary_metrics, f, ensure_ascii=False, indent=2, default=str)

print(f'Laporan evaluasi disimpan ke: {OUTPUT_PATH}')
print()
print('=== RINGKASAN METRIK (siap copy ke laporan) ===')
print(f'  Akurasi kategori   : {category_accuracy*100:.2f}%')
print(f'  Urgency consistency: {exact_rate:.1f}% exact agreement')
print(f'  Akurasi sentimen   : {sentiment_accuracy*100:.2f}% (label sintetis)')
print(f'  Manual routing     : {routing_pct:.1f}%')
